### Binärer Baum

[Video](https://youtu.be/afQKmjq2IY0)

Bei der Liste, dem Keller und der Schlange hat jedes Element höchstens einen Nachfolger. In einem Baum kann ein Element mehrere
Nachfolger haben. So lassen sich zum Beispiel Ordner auf der Festplatte, ein Turnierplan oder die Aufrufe bei den Türmen von Hanoi darstellen.

1. Der binäre Baum
2. Implementation mittels verzeigerter Knoten
3. Bäume mit Rekursion bearbeiten
4. Tiefensuche
5. Breitensuche

## 1. Der binäre Baum

Ein *binärer Baum* ist entweder leer oder besteht aus einem Knoten, dem zwei binäre Bäume zugeordnet sind. Dieser heißt dann der
*Elternknoten* des linken bzw. rechten Teilbaums. Ein Knoten ohne einen Elternknoten heißt *Wurzel*. Die Knoten, die x als Elternknoten
haben, sind *Kinder* von x. Knoten ohne Kinder heißen *Blätter*.

<img src='heapsort1.png' width='600'>

In den Knoten eines Baums können beliebige Objekte gespeichert werden.

<img src='bild2.png'>

Dies entspricht dem arithmetischen Ausdruck: $a \cdot (b + c)$

## 2. Implementation mittels verzeigerter Knoten

Jeder Knoten speichert einen Wert und zwei Zeiger auf seine Kinder. Fehlt ein Kind, steht dort `None`.

<img src='bild4.png' width='300'>

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

Die Assoziation bei der Klasse `Node` verweist auf sich selbst. Wie die Liste ist der Baum eine rekursive Datenstruktur.

<img src='Node.png' width='300'>

#### Beispiel

Wir erzeugen folgenden Baum. Dafür bauen wir erst die Teilbäume und setzen sie dann zusammen.

<img src='bild8a.png' width='200'>

In [ ]:
b = Node('b',Node('d'),Node('e'))
c = Node('c',Node('f'),None)
a = Node('a',b,c)

Ein zweites Beispiel mit Zahlen. Gezeichnet finden wir diesen Baum in Abschnitt 4.

In [ ]:
b3 = Node(3,Node(11),Node(12))
b7 = Node(7,Node(13),None)
b14 = Node(14,b3,b7)
b9 = Node(9,None,Node(23))
b10 = Node(10,Node(12),Node(16))
b6 = Node(6,b9,b10)
baum2 = Node(1,b14,b6)

Zur einfachen Visualisierung nutzen wir die Funktion `baumString`. Wenn man die Ausgabe um 90 Grad im Uhrzeigersinn dreht, zeigt sich die
Struktur des Baums. Die Anzahl der Punkte gibt die Ebene an.

In [ ]:
def baumString(node):
    def dfs(node,tiefe=0):
        s = ""
        punkte = "." * tiefe
        if node.right:
            s = s + dfs(node.right,tiefe + 1)
        s = s + punkte + str(node.val) + "\n"
        if node.left:
            s = s + dfs(node.left,tiefe + 1)
        return s

    s = dfs(node)
    if len(s) > 0:
        s = s[:-1]
    return s

In [ ]:
print(baumString(baum2))

## 3. Bäume mit Rekursion bearbeiten

Einen Baum *traversieren* bedeutet, jeden seiner Knoten besuchen. Der Baum ist eine rekursive Datenstruktur: Die Teilbäume eines Baums sind
selbst wieder Bäume. Deshalb können wir mit einer rekursiven Funktion alle Knoten des Baums besuchen.

- **Basisfall:** Der Baum ist leer, es ist nichts zu tun.
- **Rekursiver Fall:** Wir bearbeiten den Knoten und rufen die Funktion für den linken und den rechten Teilbaum auf.

In [ ]:
def dfs(node):
    if node is None: return   # Rekursionsbremse
    # do something
    dfs(node.left)
    dfs(node.right)
    return

Nach diesem Muster können wir viele Aufgaben lösen. Die Funktion `anzahl` zählt die Knoten, die Funktion `ebenen` bestimmt, wie viele Ebenen der Baum hat.

In [ ]:
def anzahl(node):
    '''
    node: Wurzel eines Baums oder None
    returns: Anzahl der Knoten
    '''
    if node is None:
        return 0
    return 1 + anzahl(node.left) + anzahl(node.right)

def ebenen(node):
    '''
    node: Wurzel eines Baums oder None
    returns: Anzahl der Ebenen
    '''
    if node is None:
        return 0
    return 1 + max(ebenen(node.left), ebenen(node.right))

print(anzahl(a), anzahl(baum2))
print(ebenen(a), ebenen(baum2))

Jeder Knoten wird genau einmal besucht, die Laufzeit ist also $O(n)$ bei $n$ Knoten.

#### Von Hand ausführen

Was eine solche Funktion ausgibt, hängt davon ab, an welcher Stelle das *do something* steht. Beim Ausführen von Hand gehen wir
Aufruf für Aufruf vor. Jeder Aufruf ist erst fertig, wenn seine beiden Teilbäume fertig sind.

Bei `f` steht die Ausgabe vor den rekursiven Aufrufen, bei `g` dahinter. Für den Baum `a` aus dem Beispiel erscheint bei `f` die Wurzel
zuerst, bei `g` zuletzt.

In [ ]:
def f(node):
    if node is None: return
    print(node.val,end=' ')
    f(node.left)
    f(node.right)

def g(node):
    if node is None: return
    g(node.left)
    g(node.right)
    print(node.val,end=' ')

f(a)
print()
g(a)

## 4. Tiefensuche

Die Funktion `f` ist eine **Tiefensuche** (depth first search, dfs): Sie läuft von einem Knoten aus erst so tief wie möglich in den linken Teilbaum
und kommt dann zurück zum rechten. Die Knoten werden in der Reihenfolge Knoten, linker Teilbaum, rechter Teilbaum besucht. Diese
Reihenfolge heißt *preorder*.

In [ ]:
f(baum2)

#### Tiefensuche mit einem Keller

Die offenen Aufrufe merkt sich der Rechner bei der Rekursion auf dem Aufrufstapel. Wir können die Tiefensuche deshalb auch ohne Rekursion mit
einem Keller organisieren. Wir laufen nach links unten und legen die rechten Teilbäume, die uns unterwegs begegnen, in den Keller.

    lege die Wurzel in den Keller
    wiederhole solange der Keller nicht leer ist
        nimm den obersten Knoten aus dem Keller
        wiederhole solange es einen Knoten gibt
            gib den Inhalt des Knotens aus
            wenn der Knoten ein rechtes Kind hat
                lege das rechte Kind in den Keller
            gehe zum linken Kind

#### Beispiel

Wir führen die Tiefensuche für den Baum `baum2` aus. Zeilenweise stehen dort die Ausgabe und der Inhalt des Kellers, nachdem das rechte Kind
(falls vorhanden) nach der Ausgabe in den Keller getan wurde. Es reicht, die Inhalte der Wurzelknoten zu notieren. Den Keller
notieren wir horizontal mit dem top-Element auf der linken Seite.

<img src='uebung.png' width='700'>

## 5. Breitensuche

Bei der **Breitensuche** (breadth first search, bfs) werden die Inhalte der Knoten ebenenweise ausgegeben. Wir reihen die linken und rechten Kinder in eine
Schlange ein.

    reihe die Wurzel in die Schlange ein
    wiederhole solange die Schlange nicht leer ist
        nimm den vordersten Knoten aus der Schlange
        gib den Inhalt des Knotens aus
        wenn der Knoten ein linkes Kind hat
            reihe das linke Kind in die Schlange ein
        wenn der Knoten ein rechtes Kind hat
            reihe das rechte Kind in die Schlange ein

#### Beispiel

Wir führen die Breitensuche für den Baum `baum2` aus. Zeilenweise stehen dort die Ausgabe und der Inhalt der Schlange, nachdem die Kinder
(falls vorhanden) nach der Ausgabe eingefügt wurden. Die Schlange notieren wir horizontal mit dem Kopf der Schlange auf der linken Seite.

<img src='breitensuche.png' width='700'>

#### Vergleich

| | Tiefensuche | Breitensuche |
|---|---|---|
| Vorgehen | erst in die Tiefe | Ebene für Ebene |
| Hilfsstruktur | Keller oder Aufrufstapel (Rekursion) | Schlange |
| Prinzip der Hilfsstruktur | LIFO | FIFO |
| findet schnell | Blätter | Knoten nah an der Wurzel |
| Laufzeit | $O(n)$ | $O(n)$ |

#### Zusammenfassung

| | |
|---|---|
| binärer Baum | entweder leer oder ein Knoten mit zwei binären Bäumen als Teilbäumen |
| Wurzel | Knoten ohne Elternknoten |
| Blatt | Knoten ohne Kinder |
| Ebene | Abstand von der Wurzel, die Wurzel liegt auf Ebene 0 |
| `Node` | `val`, `left`, `right`, ein fehlendes Kind ist `None` |
| `baumString` | stellt einen Baum als Text dar |
| traversieren | jeden Knoten besuchen, $O(n)$ |
| Rekursion auf Bäumen | Basisfall: leerer Baum, rekursiver Fall: Knoten bearbeiten, Teilbäume aufrufen |
| Tiefensuche | erst so tief wie möglich, rekursiv oder mit einem Keller |
| Breitensuche | Ebene für Ebene, mit einer Schlange |